# Student Academic Performance Analysis Using Python and Excel
**College Mini-Project in Data Analytics and Visualization (DAV)**

### Project Aim:
To systematically analyze student academic performance using Python and Excel by implementing end-to-end data processing: data ingestion, cleaning, exploratory data analysis (EDA), descriptive statistics, correlation evaluation, rich visual analytics, and predictive modeling using Scikit-Learn Linear Regression.

In [ ]:
# CELL 1: Environment Setup & Library Imports
# Purpose: Import required scientific, statistical, visual, and Excel processing libraries.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import joblib

# Fix random seed for exact reproducibility across all runs
np.random.seed(42)

# Set aesthetic style for Seaborn/Matplotlib charts
sns.set_theme(style="whitegrid", font_scale=1.1)
os.makedirs("charts", exist_ok=True)
print("✅ Cell 1 Complete: All libraries imported and environment initialized successfully!")

In [ ]:
# CELL 2: Synthetic Academic Dataset Creation
# Purpose: Generate realistic student academic records (320 students) with deliberate data quality anomalies (duplicates, missing, invalid).

n_students = 320

student_ids = [f"STU{1001 + i}" for i in range(n_students)]
genders = np.random.choice(["Male", "Female"], size=n_students, p=[0.52, 0.48])

# Generate academic features with realistic real-world bounds and distributions
attendance = np.clip(np.random.normal(loc=76, scale=12, size=n_students), 40, 100).round(1)
study_hours = np.clip(np.random.normal(loc=4.5, scale=1.8, size=n_students), 0.5, 12).round(1)

# Assignment scores influenced by study hours and attendance
assignment_score = np.clip(
    0.35 * attendance + 3.2 * study_hours + np.random.normal(0, 7, size=n_students) + 30, 35, 100
).round(1)

# Internal marks influenced by assignment and attendance
internal_marks = np.clip(
    0.45 * assignment_score + 0.25 * attendance + 1.5 * study_hours + np.random.normal(0, 6, size=n_students) + 10, 30, 100
).round(1)

# Previous semester performance
previous_marks = np.clip(np.random.normal(loc=70, scale=12, size=n_students), 40, 98).round(1)

# Final examination marks based on underlying multi-variable academic relationship
final_marks = np.clip(
    0.22 * attendance + 1.8 * study_hours + 0.25 * assignment_score + 0.30 * internal_marks + 0.20 * previous_marks + np.random.normal(0, 4.5, size=n_students) - 8,
    25, 100
).round(1)

# Create base DataFrame
df_raw = pd.DataFrame({
    "Student_ID": student_ids,
    "Gender": genders,
    "Attendance_Percentage": attendance,
    "Study_Hours": study_hours,
    "Assignment_Score": assignment_score,
    "Internal_Marks": internal_marks,
    "Previous_Semester_Marks": previous_marks,
    "Final_Marks": final_marks
})

# Inject intentional data quality issues for demonstration:
# 1. Add 6 duplicate records
duplicates = df_raw.iloc[[12, 45, 88, 120, 175, 230]].copy()
df_raw = pd.concat([df_raw, duplicates], ignore_index=True)

# 2. Inject missing values (NaN)
for idx in [15, 62, 140, 210]:
    df_raw.loc[idx, "Attendance_Percentage"] = np.nan
for idx in [33, 95, 180]:
    df_raw.loc[idx, "Study_Hours"] = np.nan
for idx in [50, 125, 270]:
    df_raw.loc[idx, "Assignment_Score"] = np.nan
for idx in [72, 199]:
    df_raw.loc[idx, "Previous_Semester_Marks"] = np.nan

# 3. Inject out-of-bounds invalid values
df_raw.loc[25, "Attendance_Percentage"] = 145.0  # Impossible attendance > 100%
df_raw.loc[82, "Study_Hours"] = -3.5             # Negative study hours
df_raw.loc[160, "Assignment_Score"] = 125.0       # Score > 100%
df_raw.loc[215, "Internal_Marks"] = -10.0         # Negative marks

# Save raw dataset
df_raw.to_csv("raw_student_data.csv", index=False)
print(f"✅ Cell 2 Complete: Synthetic Raw Dataset generated with {len(df_raw)} records (including anomalies) and saved to 'raw_student_data.csv'.")

In [ ]:
# CELL 3: Exploratory Data Inspection
# Purpose: Inspect first & last records, dimensions, schema types, missing values, duplicates, and preliminary summary.

print("--- 1. First 5 Records of Raw Data ---")
display(df_raw.head())

print("\n--- 2. Last 5 Records of Raw Data ---")
display(df_raw.tail())

print("\n--- 3. Dataset Dimensions ---")
print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")

print("\n--- 4. Schema and Data Types ---")
df_raw.info()

print("\n--- 5. Missing Values Per Column ---")
print(df_raw.isnull().sum())

print("\n--- 6. Duplicate Records Count ---")
print(f"Duplicates found: {df_raw.duplicated().sum()}")

print("\n--- 7. Statistical Summary (Raw Data) ---")
display(df_raw.describe().round(2))
print("✅ Cell 3 Complete: Data exploration completed.")

In [ ]:
# CELL 4: Data Cleaning and Preprocessing Pipeline
# Purpose: Remove duplicates, sanitize invalid range values, impute missing values with median, verify types, and evaluate outliers with IQR.

df_clean = df_raw.copy()

# Step A: Remove exact duplicate rows
dups_removed = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"[Step A] Dropped {dups_removed} duplicate records.")

# Step B: Identify and flag out-of-range invalid academic values
df_clean.loc[(df_clean["Attendance_Percentage"] < 0) | (df_clean["Attendance_Percentage"] > 100), "Attendance_Percentage"] = np.nan
df_clean.loc[(df_clean["Study_Hours"] < 0) | (df_clean["Study_Hours"] > 16), "Study_Hours"] = np.nan
df_clean.loc[(df_clean["Assignment_Score"] < 0) | (df_clean["Assignment_Score"] > 100), "Assignment_Score"] = np.nan
df_clean.loc[(df_clean["Internal_Marks"] < 0) | (df_clean["Internal_Marks"] > 100), "Internal_Marks"] = np.nan
df_clean.loc[(df_clean["Previous_Semester_Marks"] < 0) | (df_clean["Previous_Semester_Marks"] > 100), "Previous_Semester_Marks"] = np.nan
print("[Step B] Invalid domain values detected and converted to NaN for imputation.")

# Step C: Impute missing values using column-wise medians
numeric_cols = ["Attendance_Percentage", "Study_Hours", "Assignment_Score", "Internal_Marks", "Previous_Semester_Marks", "Final_Marks"]
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        col_median = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(col_median)
print("[Step C] Missing values successfully imputed with median values.")

# Step D: Type conversion and rounding
df_clean["Student_ID"] = df_clean["Student_ID"].astype(str)
df_clean["Gender"] = df_clean["Gender"].astype(str)
for col in numeric_cols:
    df_clean[col] = df_clean[col].astype(float).round(2)
print("[Step D] Types confirmed and numerical columns rounded to 2 decimal places.")

# Step E: Outlier Analysis via Interquartile Range (IQR)
print("\n[Step E] Outlier Evaluation via Interquartile Range (IQR):")
for col in numeric_cols:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
    print(f" - {col:<24}: IQR={iqr:.2f}, Lower={lower_bound:.2f}, Upper={upper_bound:.2f} | Outliers={len(outliers)}")

# Validation check
assert df_clean.isnull().sum().sum() == 0, "Validation Error: Remaining missing values detected!"
assert df_clean.duplicated().sum() == 0, "Validation Error: Remaining duplicates detected!"

# Save cleaned dataset
df_clean.to_csv("cleaned_student_data.csv", index=False)
print(f"\n✅ Cell 4 Complete: Cleaned dataset saved as 'cleaned_student_data.csv' ({len(df_clean)} records).")

In [ ]:
# CELL 5: Descriptive Statistical Analysis & Academic Categorization
# Purpose: Calculate Mean, Median, Mode, Std Dev, Variance, Min, Max, Range, Q1, Q3, and group into 4 Performance Categories.

stats_data = []
for col in numeric_cols:
    col_s = df_clean[col]
    stats_data.append({
        "Variable": col,
        "Mean": round(col_s.mean(), 2),
        "Median": round(col_s.median(), 2),
        "Mode": round(col_s.mode().iloc[0], 2),
        "Std_Dev": round(col_s.std(), 2),
        "Variance": round(col_s.var(), 2),
        "Min": round(col_s.min(), 2),
        "Max": round(col_s.max(), 2),
        "Range": round(col_s.max() - col_s.min(), 2),
        "Q1 (25%)": round(col_s.quantile(0.25), 2),
        "Q3 (75%)": round(col_s.quantile(0.75), 2)
    })

df_stats = pd.DataFrame(stats_data)
print("=== Descriptive Statistical Summary ===")
display(df_stats)

# Performance Categories definition
def categorize_marks(score):
    if score >= 85.0:
        return "Excellent (85-100)"
    elif score >= 70.0:
        return "Good (70-84.99)"
    elif score >= 50.0:
        return "Average (50-69.99)"
    else:
        return "Needs Improvement (<50)"

df_clean["Performance_Category"] = df_clean["Final_Marks"].apply(categorize_marks)

cat_order = ["Excellent (85-100)", "Good (70-84.99)", "Average (50-69.99)", "Needs Improvement (<50)"]
cat_counts = df_clean["Performance_Category"].value_counts().reindex(cat_order, fill_value=0)
cat_pcts = ((cat_counts / len(df_clean)) * 100).round(2)
cat_avg_marks = df_clean.groupby("Performance_Category")["Final_Marks"].mean().reindex(cat_order, fill_value=0).round(2)

df_categories = pd.DataFrame({
    "Performance_Category": cat_order,
    "Student_Count": cat_counts.values,
    "Percentage (%)": cat_pcts.values,
    "Average_Final_Marks": cat_avg_marks.values
})

print("\n=== Academic Performance Distribution ===")
display(df_categories)
print("✅ Cell 5 Complete: Statistical measures & performance categorization calculated.")

In [ ]:
# CELL 6: Correlation Analysis
# Purpose: Compute Pearson correlation matrix, analyze feature relationships with Final_Marks, and rank them.

df_corr = df_clean[numeric_cols].corr().round(4)
print("=== Pearson Correlation Matrix ===")
display(df_corr)

corr_with_final = df_corr["Final_Marks"].drop("Final_Marks").sort_values(ascending=False)
print("\n=== Correlation of Predictor Features with Final Examination Marks ===")
for feat, r_val in corr_with_final.items():
    print(f" - {feat:<25}: r = {r_val:>6.4f}")

print("\nKey Insight: Assignment Score and Internal Marks exhibit the strongest positive correlations with Final Marks.")
print("✅ Cell 6 Complete: Correlation analysis calculated.")

In [ ]:
# CELL 7: Comprehensive Data Visualization (7 High-Resolution Charts)
# Purpose: Create and export all required visualizations to PNG format with professional formatting.

# 1. Histogram of Final Marks
plt.figure(figsize=(8, 5))
sns.histplot(df_clean["Final_Marks"], kde=True, bins=15, color="#1e3d59", edgecolor="white")
plt.axvline(df_clean["Final_Marks"].mean(), color="#ff6e40", linestyle="--", linewidth=2, label=f"Mean: {df_clean['Final_Marks'].mean():.2f}")
plt.axvline(df_clean["Final_Marks"].median(), color="#17b978", linestyle="-.", linewidth=2, label=f"Median: {df_clean['Final_Marks'].median():.2f}")
plt.title("Chart 1: Distribution of Final Examination Marks", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Final Examination Marks", fontsize=11)
plt.ylabel("Frequency of Students", fontsize=11)
plt.legend()
plt.tight_layout()
plt.savefig("charts/01_final_marks_distribution.png", dpi=300)
plt.show()

# 2. Bar Chart of Performance Categories
plt.figure(figsize=(8, 5))
bars = plt.bar(df_categories["Performance_Category"], df_categories["Average_Final_Marks"], color=["#2e7d32", "#1976d2", "#f57c00", "#c62828"], edgecolor="black", alpha=0.85)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 1.2, f"{yval:.1f}", ha='center', va='bottom', fontsize=10, fontweight="bold")
plt.title("Chart 2: Average Marks by Performance Category", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Performance Category", fontsize=11)
plt.ylabel("Average Final Marks", fontsize=11)
plt.ylim(0, 105)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig("charts/02_performance_categories_barchart.png", dpi=300)
plt.show()

# 3. Scatter Plot: Study Hours vs Final Marks
plt.figure(figsize=(8, 5))
sns.regplot(data=df_clean, x="Study_Hours", y="Final_Marks", scatter_kws={'alpha':0.6, 'color': '#0d47a1'}, line_kws={'color': '#d50000', 'linewidth': 2})
plt.title("Chart 3: Daily Study Hours vs. Final Examination Marks", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Daily Study Hours", fontsize=11)
plt.ylabel("Final Examination Marks", fontsize=11)
plt.tight_layout()
plt.savefig("charts/03_study_hours_vs_final_marks.png", dpi=300)
plt.show()

# 4. Scatter Plot: Attendance vs Final Marks
plt.figure(figsize=(8, 5))
sns.regplot(data=df_clean, x="Attendance_Percentage", y="Final_Marks", scatter_kws={'alpha':0.6, 'color': '#00695c'}, line_kws={'color': '#ff6d00', 'linewidth': 2})
plt.title("Chart 4: Attendance Percentage vs. Final Examination Marks", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Attendance Percentage (%)", fontsize=11)
plt.ylabel("Final Examination Marks", fontsize=11)
plt.tight_layout()
plt.savefig("charts/04_attendance_vs_final_marks.png", dpi=300)
plt.show()

# 5. Heatmap of Correlation Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(df_corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Chart 5: Correlation Heatmap of Academic Features", fontsize=13, fontweight="bold", pad=10)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig("charts/05_correlation_heatmap.png", dpi=300)
plt.show()

# 6. Box Plot of Academic Variables
plt.figure(figsize=(9, 5))
box_vars = ["Attendance_Percentage", "Assignment_Score", "Internal_Marks", "Previous_Semester_Marks", "Final_Marks"]
sns.boxplot(data=df_clean[box_vars], palette="Set2")
plt.title("Chart 6: Box Plot of Key Academic Variables", fontsize=13, fontweight="bold", pad=10)
plt.ylabel("Score / Percentage", fontsize=11)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig("charts/06_academic_variables_boxplot.png", dpi=300)
plt.show()

print("✅ Cell 7 Complete: Charts 1 through 6 plotted and saved to 'charts/' directory.")

In [ ]:
# CELL 8: Machine Learning - Linear Regression Pipeline
# Purpose: Feature selection, train-test splitting (80/20), model training, evaluation (MAE, MSE, RMSE, R²), and Chart 7.

features = ["Attendance_Percentage", "Study_Hours", "Assignment_Score", "Internal_Marks", "Previous_Semester_Marks"]
target = "Final_Marks"

X = df_clean[features]
y = df_clean[target]

# 80% Train, 20% Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training instances: {len(X_train)} | Testing instances: {len(X_test)}")

# Model Initialization and Training
model = LinearRegression()
model.fit(X_train, y_train)

# Model Evaluation on Test Set
y_pred_test = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_test)

df_metrics = pd.DataFrame({
    "Evaluation_Metric": ["Mean Absolute Error (MAE)", "Mean Squared Error (MSE)", "Root Mean Squared Error (RMSE)", "R-squared Score (R²)"],
    "Value": [round(mae, 4), round(mse, 4), round(rmse, 4), round(r2, 4)],
    "Interpretation": [
        "Average absolute prediction error in marks",
        "Mean squared penalty for large deviations",
        "Standard deviation of residuals in exam marks",
        f"Model accounts for {r2*100:.2f}% of final marks variance on test data"
    ]
})
print("=== Regression Evaluation Metrics ===")
display(df_metrics)

# Regression Equation Parameters
df_coeffs = pd.DataFrame({
    "Parameter": ["Intercept (β₀)"] + [f"β ({col})" for col in features],
    "Weight": [round(model.intercept_, 4)] + [round(c, 4) for c in model.coef_]
})
print("\n=== Fitted Regression Equation Parameters ===")
display(df_coeffs)

# Chart 7: Actual vs Predicted Marks Plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_test, color="#1565c0", alpha=0.75, edgecolors="k", s=50, label="Student Test Samples")
min_val = min(y_test.min(), y_pred_test.min()) - 2
max_val = max(y_test.max(), y_pred_test.max()) + 2
plt.plot([min_val, max_val], [min_val, max_val], color="#c62828", linestyle="--", linewidth=2, label="Ideal Fit (Actual = Predicted)")
plt.title(f"Chart 7: Actual vs. Predicted Marks (R² = {r2:.3f}, RMSE = {rmse:.2f})", fontsize=13, fontweight="bold", pad=10)
plt.xlabel("Actual Final Examination Marks", fontsize=11)
plt.ylabel("Predicted Final Examination Marks", fontsize=11)
plt.xlim(min_val, max_val)
plt.ylim(min_val, max_val)
plt.legend()
plt.tight_layout()
plt.savefig("charts/07_actual_vs_predicted_marks.png", dpi=300)
plt.show()

# Save model
joblib.dump(model, "student_performance_model.pkl")
print("✅ Cell 8 Complete: Model trained, evaluated, Chart 7 saved, and model exported.")

In [ ]:
# CELL 9: Interactive New Student Prediction Function
# Purpose: Accept custom student inputs, validate ranges, and forecast estimated final marks.

def predict_student_performance(attendance_pct, daily_study_hrs, assignment_sc, internal_mk, prev_sem_mk):
    """
    Accepts student academic parameters, validates domains, and outputs predicted marks.
    """
    # Domain Validations
    if not (0 <= attendance_pct <= 100):
        raise ValueError(f"Invalid Attendance: {attendance_pct}%. Range must be [0, 100].")
    if not (0 <= daily_study_hrs <= 16):
        raise ValueError(f"Invalid Study Hours: {daily_study_hrs} hrs. Range must be [0, 16].")
    if not (0 <= assignment_sc <= 100):
        raise ValueError(f"Invalid Assignment Score: {assignment_sc}. Range must be [0, 100].")
    if not (0 <= internal_mk <= 100):
        raise ValueError(f"Invalid Internal Marks: {internal_mk}. Range must be [0, 100].")
    if not (0 <= prev_sem_mk <= 100):
        raise ValueError(f"Invalid Previous Semester Marks: {prev_sem_mk}. Range must be [0, 100].")
    
    input_df = pd.DataFrame([{
        "Attendance_Percentage": attendance_pct,
        "Study_Hours": daily_study_hrs,
        "Assignment_Score": assignment_sc,
        "Internal_Marks": internal_mk,
        "Previous_Semester_Marks": prev_sem_mk
    }])
    
    prediction = model.predict(input_df)[0]
    bounded_prediction = round(float(np.clip(prediction, 0, 100)), 2)
    
    print("="*55)
    print("       STUDENT FINAL PERFORMANCE PREDICTION")
    print("="*55)
    print(f"Attendance Percentage     : {attendance_pct}%")
    print(f"Daily Study Hours         : {daily_study_hrs} hrs/day")
    print(f"Assignment Score          : {assignment_sc}/100")
    print(f"Internal Assessment Marks : {internal_mk}/100")
    print(f"Previous Semester Marks   : {prev_sem_mk}/100")
    print("-"*55)
    print(f"🎯 Estimated Final Marks  : {bounded_prediction}/100")
    print("="*55)
    print("Note: This score is a statistical regression estimate based on historical trends, not a guaranteed outcome.")
    return bounded_prediction

# Example demonstration
test_score = predict_student_performance(85.0, 5.0, 88.0, 82.0, 78.0)
print("✅ Cell 9 Complete: Prediction function tested.")

In [ ]:
# CELL 10: Multi-Sheet Excel Workbook Generation (OpenPyXL)
# Purpose: Create 'Student_Academic_Performance_Analysis.xlsx' with 7 formatted worksheets, formulas, and auto-filters.

wb = openpyxl.Workbook()
wb.remove(wb.active)

# Styling Definitions
header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
header_navy = PatternFill(start_color="1B365D", end_color="1B365D", fill_type="solid")
header_teal = PatternFill(start_color="005A70", end_color="005A70", fill_type="solid")
title_font = Font(name="Calibri", size=13, bold=True, color="1B365D")
regular_font = Font(name="Calibri", size=11)
bold_font = Font(name="Calibri", size=11, bold=True)
thin_border = Border(left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'), top=Side(style='thin', color='D9D9D9'), bottom=Side(style='thin', color='D9D9D9'))
double_bottom = Border(left=Side(style='thin', color='D9D9D9'), right=Side(style='thin', color='D9D9D9'), top=Side(style='thin', color='D9D9D9'), bottom=Side(style='double', color='1B365D'))

def format_sheet(ws, title_text, fill_color=header_navy):
    ws.insert_rows(1, 2)
    ws["A1"] = title_text
    ws["A1"].font = title_font
    header_row = 3
    for c in range(1, ws.max_column + 1):
        cell = ws.cell(row=header_row, column=c)
        cell.font = header_font
        cell.fill = fill_color
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for r in range(4, ws.max_row + 1):
        for c in range(1, ws.max_column + 1):
            cell = ws.cell(row=r, column=c)
            cell.font = regular_font
            cell.border = thin_border
            cell.alignment = Alignment(horizontal="right" if isinstance(cell.value, (int, float)) else "left", vertical="center")
    for col in ws.columns:
        max_l = max(len(str(c.value or "")) for c in col)
        ws.column_dimensions[get_column_letter(col[0].column)].width = max(max_l + 4, 13)
    ws.freeze_panes = "A4"

# 1. Raw_Data
ws1 = wb.create_sheet(title="Raw_Data")
for row in [df_raw.columns.tolist()] + df_raw.values.tolist(): ws1.append(row)
format_sheet(ws1, "Raw Student Academic Records (Contains Missing, Duplicates & Invalid Records)")

# 2. Cleaned_Data
ws2 = wb.create_sheet(title="Cleaned_Data")
for row in [df_clean.columns.tolist()] + df_clean.values.tolist(): ws2.append(row)
format_sheet(ws2, "Cleaned Student Academic Records (320 Cleaned Records)", header_teal)
# Add Excel Formulas for Average and Count
max_r = ws2.max_row
ws2.cell(row=max_r+2, column=2, value="AVERAGE (Excel Formula)").font = bold_font
ws2.cell(row=max_r+3, column=2, value="COUNT (Excel Formula)").font = bold_font
for c_idx, c_name in enumerate(df_clean.columns, 1):
    col_l = get_column_letter(c_idx)
    if c_name in numeric_cols:
        c_avg = ws2.cell(row=max_r+2, column=c_idx, value=f"=AVERAGE({col_l}4:{col_l}{max_r})")
        c_avg.font = bold_font; c_avg.number_format = "0.00"; c_avg.border = double_bottom
        c_cnt = ws2.cell(row=max_r+3, column=c_idx, value=f"=COUNT({col_l}4:{col_l}{max_r})")
        c_cnt.font = bold_font; c_cnt.border = double_bottom

# 3. Statistical_Summary
ws3 = wb.create_sheet(title="Statistical_Summary")
for row in [df_stats.columns.tolist()] + df_stats.values.tolist(): ws3.append(row)
format_sheet(ws3, "Descriptive Statistics of Academic Variables")

# 4. Correlation_Matrix
ws4 = wb.create_sheet(title="Correlation_Matrix")
corr_exp = df_corr.reset_index().rename(columns={"index": "Variable"})
for row in [corr_exp.columns.tolist()] + corr_exp.values.tolist(): ws4.append(row)
format_sheet(ws4, "Pearson Correlation Matrix of Features", header_teal)

# 5. Performance_Categories
ws5 = wb.create_sheet(title="Performance_Categories")
for row in [df_categories.columns.tolist()] + df_categories.values.tolist(): ws5.append(row)
format_sheet(ws5, "Performance Categories Distribution")

# 6. Model_Evaluation
ws6 = wb.create_sheet(title="Model_Evaluation")
ws6.append(["Regression Weights & Parameters", ""]); ws6.append(["Parameter Name", "Coefficient Value"])
for _, r in df_coeffs.iterrows(): ws6.append(r.tolist())
ws6.append([]); ws6.append(["Model Goodness-of-Fit Metrics", "", ""])
ws6.append(["Metric", "Score", "Description"])
for _, r in df_metrics.iterrows(): ws6.append(r.tolist())
format_sheet(ws6, "Linear Regression Equation & Evaluation Metrics", header_navy)

# 7. Predictions
ws7 = wb.create_sheet(title="Predictions")
df_pred_tab = X_test.copy()
df_pred_tab["Actual_Final_Marks"] = y_test.values
df_pred_tab["Predicted_Final_Marks"] = np.round(y_pred_test, 2)
df_pred_tab["Residual_Error"] = np.round(df_pred_tab["Actual_Final_Marks"] - df_pred_tab["Predicted_Final_Marks"], 2)
pred_exp = df_pred_tab.reset_index().rename(columns={"index": "Sample_Index"})
for row in [pred_exp.columns.tolist()] + pred_exp.values.tolist(): ws7.append(row)
format_sheet(ws7, "Actual vs Predicted Marks (Test Dataset)", header_teal)

excel_filename = "Student_Academic_Performance_Analysis.xlsx"
wb.save(excel_filename)
print(f"✅ Cell 10 Complete: Professional Excel Workbook '{excel_filename}' successfully created with 7 worksheets and formulas!")

In [ ]:
# CELL 11: Automated Project Summary Report
# Purpose: Automatically summarize the entire experimental findings directly from calculated objects.

print("="*75)
print("             STUDENT ACADEMIC PERFORMANCE ANALYSIS - SUMMARY")
print("="*75)
print(f"Total Raw Records Ingested     : {len(df_raw)}")
print(f"Duplicate Records Cleared      : {dups_removed}")
print(f"Total Cleaned Student Records  : {len(df_clean)}")
print(f"Average Final Examination Marks: {df_clean['Final_Marks'].mean():.2f}")
print(f"Average Attendance Percentage  : {df_clean['Attendance_Percentage'].mean():.2f}%")
print(f"Average Daily Study Hours      : {df_clean['Study_Hours'].mean():.2f} hrs/day")
print("-"*75)
print("Academic Category Distribution:")
for _, r in df_categories.iterrows():
    print(f"  • {r['Performance_Category']:<26}: {r['Student_Count']:>3} students ({r['Percentage (%)']:>5.2f}%) | Avg Marks: {r['Average_Final_Marks']}")
print("-"*75)
print("Correlations with Final Marks (Ranked):")
for var, val in corr_with_final.items():
    print(f"  • {var:<26}: r = {val:>6.4f}")
print("-"*75)
print("Linear Regression Evaluation (Test Set = 20%):")
print(f"  • Mean Absolute Error (MAE)   : {mae:.4f}")
print(f"  • Mean Squared Error (MSE)    : {mse:.4f}")
print(f"  • Root Mean Squared Error     : {rmse:.4f}")
print(f"  • R-squared Score (R²)        : {r2:.4f} ({r2*100:.2f}% variance explained)")
print("="*75)
print("✅ Cell 11 Complete: Automated results summary displayed.")

In [ ]:
# CELL 12: Project Artifacts Archiving and Download (Google Colab Code)
# Purpose: Bundle all outputs (CSV, Excel, PNG charts, trained model) into a ZIP archive and provide download triggers.

import shutil

# Create a clean zip archive of all project artifacts
archive_name = "Student_Academic_Performance_Project_Files"
shutil.make_archive(archive_name, 'zip', root_dir=".", base_dir=".")
print(f"📦 Archive '{archive_name}.zip' created containing all project deliverables.")

# Google Colab Download Code (Uncomment to automatically trigger browser download in Colab):
try:
    from google.colab import files
    print("Triggering download of Excel Workbook...")
    files.download("Student_Academic_Performance_Analysis.xlsx")
    print("Triggering download of Cleaned Dataset...")
    files.download("cleaned_student_data.csv")
    print("Triggering download of Project Zip Bundle...")
    files.download(f"{archive_name}.zip")
except ImportError:
    print("Running in local environment: All files saved locally in current working directory.")
print("✅ Cell 12 Complete: Project deliverables archived and download pipeline ready.")